In [1]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

In [5]:
def banapresso_menu():
    options = Options()
    options.add_argument("--start-maximized")
    url = "https://www.banapresso.com/"
    driver = webdriver.Chrome(options=options)
    driver.get(url)
    time.sleep(2)

    action = ActionChains(driver)

    # 메뉴 > 주문하기
    first_tag = driver.find_element(By.CSS_SELECTOR, "#wrap > header > div > ul > li:nth-child(1) > a")
    second_tag = driver.find_element(By.CSS_SELECTOR, "#wrap > header > div > ul > li:nth-child(1) > ul > li:nth-child(2) > a")
    action.move_to_element(first_tag).move_to_element(second_tag).click().perform()
    time.sleep(3)
    print("메뉴 > 주문하기 이동 성공!")

    # html로 파서
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    sections = soup.select('section')
    
    data = {}

    # 데이터 추출
    for section in sections:
        try:
            categories = section.select_one("h2").text
            menus = section.select("strong")
            data[categories] = [menu.text for menu in menus]

        except:
            continue

    # 엑셀 파일 쓰기 모드로 열기
    with pd.ExcelWriter('0629_banapresso.xlsx') as writer:
        # 딕셔너리 순회하며 시트 저장
        for category, menu_list in data.items():
            df = pd.DataFrame({ '메뉴': menu_list })
            df.to_excel(writer, sheet_name=category, index=False)

    driver.quit()

In [6]:
banapresso_menu()

메뉴 > 주문하기 이동 성공!


***

In [26]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support import expected_conditions as EC


In [27]:
def extract_current_page(driver, wait):
    data = []
    try:
        menu_table = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, '//*[@id="menu_list"]')
            )
        )

    except TimeoutException:
        return []
    
    rows = menu_table.find_elements(By.CLASS_NAME, "cont_gallery_list_box")

    for row in rows:
        menu_name = row.find_element(By.CSS_SELECTOR, "div.text.text1 b").text
        menu_eng_name = row.find_element(By.CSS_SELECTOR, "div.cont_text_inner.cont_text_info > div.text.text1").text
        comment = row.find_element(By.CSS_SELECTOR, "div.text.text2").text

        data.append({
            "메뉴명": menu_name,
            "영어명": menu_eng_name,
            "설명": comment
        })

    return data

In [33]:
def fetch_mega(max_page=10):
    options = Options()
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    categories = {
        "음료": 1,
        "푸드": 2,
        "상품": 3,
    }

    data = {}

    try:
        action = ActionChains(driver)
        
        for category, idx in categories.items():
            driver.get("https://www.mega-mgccoffee.com/")
            time.sleep(2)

            first_tag = driver.find_element(By.CSS_SELECTOR, "body > div.wrap > div.head_wrap > div > div.head_menu_wrap > div.head_menu > ul > li:nth-child(2) > a.pc")

            second_tag = driver.find_element(By.CSS_SELECTOR,
                f"body > div.wrap > div.head_wrap > div > div.head_menu_wrap > div.head_menu > ul > li:nth-child(2) > div > ul > li:nth-child({idx}) > a")
            action.move_to_element(first_tag).move_to_element(second_tag).click().perform()
            time.sleep(2)

            all_data = []

            for page in range(1, max_page + 1):
                page_data = extract_current_page(driver, wait)

                if not page_data:
                    print(f"{page}페이지 데이터가 없어 종료합니다.")
                    break
                
                all_data.extend(page_data)
                print(f"{page}페이지 크롤링 완료: {len(page_data)}개의 메뉴")
                
                try:
                    next_btn = driver.find_element(By.CSS_SELECTOR,  f'a[data-page="{page + 1}"]')
                    next_btn.click()
                    time.sleep(2)
                    
                except Exception as e:
                    print("다음 페이지 이동 실패:", e)
                    break
            data[category] = all_data

        with pd.ExcelWriter('0629_mega_menu.xlsx') as writer:
            for category, menu_list in data.items():
                df = pd.DataFrame(menu_list)
                df.to_excel(writer, sheet_name=category, index=False)
        
        print("엑셀 저장 완료!")

    finally:
        driver.quit()

In [34]:
fetch_mega()


1페이지 크롤링 완료: 20개의 메뉴
2페이지 크롤링 완료: 20개의 메뉴
3페이지 크롤링 완료: 20개의 메뉴
4페이지 크롤링 완료: 20개의 메뉴
5페이지 크롤링 완료: 20개의 메뉴
6페이지 크롤링 완료: 20개의 메뉴
7페이지 크롤링 완료: 20개의 메뉴
8페이지 크롤링 완료: 20개의 메뉴
9페이지 크롤링 완료: 2개의 메뉴
다음 페이지 이동 실패: Message: no such element: Unable to locate element: {"method":"css selector","selector":"a[data-page="10"]"}
  (Session info: chrome=149.0.7827.200); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff74e4f3fa5+14925]
	chromedriver!GetHandleVerifier [0x7ff74e4f4000+14980]
	chromedriver!(No symbol) [0x7ff74e03793d]
	chromedriver!(No symbol) [0x7ff74e091aad]
	chromedriver!(No symbol) [0x7ff74e091dac]
	chromedriver!(No symbol) [0x7ff74e0e27d7]
	chromedriver!(No symbol) [0x7ff74e0df39b]
	chromedriver!(No symbol) [0x7ff74e08401c]
	chromedriver!(No symbol) [0x7ff74e084f43]
	chromedriver!GetHandleVerifier [0x7ff74ead7591+5f7f11]
	chromedriver!GetHandleVeri